# Setup Agentic Information Retrieval
Questo notebook è configurato per l'esecuzione su Google Colab. Assicurati di impostare l'acceleratore hardware su **T4 GPU** (Runtime -> Change runtime type).

In [ ]:
# 1. Verifica che la GPU sia attiva
!nvidia-smi

# 2. Installa le dipendenze core
!pip install -q langchain langgraph langchain-community huggingface-hub pydantic duckduckgo-search ddgs wikipedia

# 3. Installa llama-cpp-python compilato con supporto CUDA per sfruttare la GPU di Colab
# Rileviamo la versione di CUDA e usiamo la libreria PRE-COMPILATA per evitare 10 minuti di build
wheel_url = "https://abetlen.github.io/llama-cpp-python/whl/cu124"
!pip install llama-cpp-python --extra-index-url {wheel_url} -q


## Clonazione del Codice Sorgente
Poiché hai caricato questo notebook da GitHub, i file Python necessari (`graph.py`, `Refiner.py`, ecc.) non sono automaticamente nel file system di Colab. Dobbiamo clonare la repository per importarli.

In [ ]:
import os

# INSERISCI QUI IL LINK ALLA TUA REPOSITORY:
REPO_URL = "https://github.com/GabCovone/AgenticAskMe"

repo_name = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    !git clone {REPO_URL}

# Spostiamoci nella cartella del progetto per permettere gli import Python
os.chdir(repo_name)
print(f"Directory di lavoro attuale: {os.getcwd()}")

## Inizializzazione del Grafo e del Modello
Grazie alla centralizzazione in `graph.py`, il modello da 14B parametri (circa 10GB) verrà scaricato e caricato in VRAM una sola volta.

In [ ]:
!git pull
from graph import build_graph

# Inizializza il grafo e scarica i pesi del LLM
app = build_graph()
print("Grafo compilato e pronto all'uso!")

## Benchmark del Sistema Multi-Agente
Eseguiamo un benchmark automatico utilizzando **HotpotQA**, il dataset gold-standard per le domande multi-hop, per calcolare l'accuratezza del nostro sistema RAG Agentico.

In [ ]:
from datasets import load_dataset
import time

# Per il benchmark multi-hop, usiamo HotpotQA (il dataset gold-standard per questo task).
# (CWQ nativo è stato deprecato da HuggingFace, HotpotQA è la sua evoluzione più stabile).
print("Scaricamento del dataset HotpotQA in corso...")
dataset = load_dataset("hotpotqa/hotpot_qa", "fullwiki", split="validation")

NUM_TESTS = 10  # Numero di domande per il benchmark (modificabile)
correct = 0

print(f"\n--- AVVIO BENCHMARK SU {NUM_TESTS} DOMANDE --- ")
for i in range(NUM_TESTS):
    sample = dataset[i]
    question = sample['question']
    gold_answer = sample['answer']
    
    print(f"\n[Test {i+1}/{NUM_TESTS}]")
    print(f"Domanda: {question}")
    print(f"Gold Answer: {gold_answer}")
    
    state = {
        "original_query": question,
        "current_query": question,
        "retrieved_context": "",
        "num_refinement": 0,
        "num_planning": 0,
        "next_node": "",
        "feedback_history": [],
        "final_answer": ""
    }
    
    final_answer = ""
    try:
        for output in app.stream(state):
            for key, value in output.items():
                if key == "planning_node" and "final_answer" in value:
                    final_answer = value["final_answer"]
    except Exception as e:
        print(f"Errore durante l'esecuzione: {e}")
        
    print(f"Risposta Agente: {final_answer}")
    
    # Metrica di valutazione semplice: Exact Match / Substring Match
    if gold_answer.lower() in final_answer.lower():
        print("Valutazione: CORRETTA ✅")
        correct += 1
    else:
        print("Valutazione: ERRATA ❌")
        
    time.sleep(1) # Pausa per non sovraccaricare le API di ricerca

accuracy = (correct / NUM_TESTS) * 100
print("\n" + "="*50)
print(f"RISULTATI BENCHMARK: {correct}/{NUM_TESTS} corrette.")
print(f"ACCURACY (Substring Match): {accuracy}%")
print("="*50 + "\n")